In [1]:
import cv2
from datetime import timedelta

# Input and output paths
input_video = "../demo/ANMR0006.mp4"  # UPDATE THIS PATH
output_video = "outputs/trimmed_demo.mp4"

# Time range: 1:50 to 2:50
start_time = timedelta(minutes=1, seconds=50).total_seconds()  # 110 seconds
end_time = timedelta(minutes=2, seconds=50).total_seconds()    # 170 seconds

# Open video
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video info: {fps} fps, {width}x{height}, {total_frames} frames")
print(f"Duration: {total_frames/fps:.2f} seconds")

# Calculate frame range
start_frame = int(start_time * fps)
end_frame = int(end_time * fps)

print(f"Trimming from frame {start_frame} to {end_frame}")

# Setup video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# Set to start frame
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

# Read and write frames
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or cap.get(cv2.CAP_PROP_POS_FRAMES) > end_frame:
        break
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()

print(f"\nDone! Trimmed video saved to: {output_video}")
print(f"Total frames extracted: {frame_count}")

Video info: 60.0 fps, 1920x1080, 31388 frames
Duration: 523.13 seconds
Trimming from frame 6600 to 10200
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Processed 540 frames...
Processed 570 frames...
Processed 600 frames...
Processed 630 frames...
Processed 660 frames...
Processed 690 frames...
Processed 720 frames...
Processed 750 frames...
Processed 780 frames...
Processed 810 frames...
Processed 840 frames...
Processed 870 frames...
Processed 900 frames...
Processed 930 frames...
Processed 960 frames...
Processed 990 frames...
Processed 1020 frames...
Processed 1050 frames...
Processed 1080 frames...
Processed 1110 frames...
Proces

In [10]:
import sys
sys.path.append('src')

from object_detection.inference.predictor import ObjectDetector
from object_detection.tracking.simple_tracker import SimpleTracker
from object_detection.utils.config import DetectionConfig
import cv2
import pandas as pd
from pathlib import Path

# Paths
video_path = "../outputs/trimmed_demo.mp4"
model_path = "../models/weights/best.pt"
output_csv = "../outputs/demo_tracking_data.csv"

print("Loading model...")
detector = ObjectDetector(config_path=None, model_path=model_path)
detector.config.device = 'cpu'

print("Initializing tracker...")
tracker = SimpleTracker(max_disappeared=30)

# Open video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

# Storage for tracking data
tracking_data = []

print(f"Processing video at {fps} fps...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run detection
    results = detector.predict(frame)
    
    # Extract detections from the wrapper format
    detections = results[0]['detections'] if results and 'detections' in results[0] else []
    
    # Update tracker
    tracked_objects = tracker.update(detections)
    
    # Store data
    timestamp = frame_count / fps
    
    for obj in tracked_objects:
        tracking_data.append({
            'frame': frame_count,
            'timestamp': timestamp,
            'track_id': obj['track_id'],
            'class': obj['class_name'],
            'confidence': obj['confidence'],
            'pixel_x': obj['bbox'][0] + obj['bbox'][2]/2,  # center x
            'pixel_y': obj['bbox'][1] + obj['bbox'][3]/2   # center y
        })
    
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames, detected {len(tracked_objects)} objects")

cap.release()

# Save to CSV
df = pd.DataFrame(tracking_data)
df.to_csv(output_csv, index=False)

print(f"\nProcessing complete!")
print(f"Total frames: {frame_count}")
print(f"Total detections: {len(tracking_data)}")
print(f"Unique tracks: {df['track_id'].nunique()}")
print(f"CSV saved to: {output_csv}")
print(f"\nFirst few rows:")
print(df.head(10))

Loading model...
Initializing tracker...
Processing video at 60.0 fps...


0: 384x640 2 persons, 128.3ms
Speed: 1.5ms preprocess, 128.3ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict6

0: 384x640 2 persons, 53.2ms
Speed: 3.5ms preprocess, 53.2ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7

0: 384x640 2 persons, 72.3ms
Speed: 1.0ms preprocess, 72.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict8

0: 384x640 2 persons, 54.5ms
Speed: 1.0ms preprocess, 54.5ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict9

0: 384x640 2 persons, 56.1ms
Speed: 1.1ms preprocess, 56.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict10

0: 384x640 2 persons, 53.4ms
Speed: 1.3ms preprocess, 53.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Result

Processed 30 frames, detected 2 objects


0: 384x640 2 persons, 54.2ms
Speed: 1.2ms preprocess, 54.2ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict39

0: 384x640 2 persons, 53.8ms
Speed: 1.3ms preprocess, 53.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict40

0: 384x640 2 persons, 52.4ms
Speed: 1.2ms preprocess, 52.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict41

0: 384x640 2 persons, 53.2ms
Speed: 1.1ms preprocess, 53.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict42

0: 384x640 2 persons, 52.4ms
Speed: 1.1ms preprocess, 52.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict43

0: 384x640 2 persons, 53.1ms
Speed: 1.1ms preprocess, 53.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Resu

Processed 60 frames, detected 2 objects


...

Processed 3540 frames, detected 2 objects


0: 384x640 2 persons, 53.0ms
Speed: 1.3ms preprocess, 53.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3548

0: 384x640 2 persons, 55.0ms
Speed: 1.1ms preprocess, 55.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3549

0: 384x640 2 persons, 54.8ms
Speed: 1.2ms preprocess, 54.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3550

0: 384x640 2 persons, 55.8ms
Speed: 1.1ms preprocess, 55.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3551

0: 384x640 2 persons, 54.9ms
Speed: 1.2ms preprocess, 54.9ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3552

0: 384x640 2 persons, 53.0ms
Speed: 1.2ms preprocess, 53.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384,

Processed 3570 frames, detected 2 objects


0: 384x640 2 persons, 48.8ms
Speed: 1.1ms preprocess, 48.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3578

0: 384x640 2 persons, 75.7ms
Speed: 1.1ms preprocess, 75.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3579

0: 384x640 2 persons, 51.9ms
Speed: 1.3ms preprocess, 51.9ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3580

0: 384x640 2 persons, 50.7ms
Speed: 1.1ms preprocess, 50.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3581

0: 384x640 2 persons, 50.5ms
Speed: 1.0ms preprocess, 50.5ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3582

0: 384x640 2 persons, 52.1ms
Speed: 1.1ms preprocess, 52.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384,

Processed 3600 frames, detected 2 objects

Processing complete!
Total frames: 3600
Total detections: 11028
Unique tracks: 50
CSV saved to: ../outputs/demo_tracking_data.csv

First few rows:
   frame  timestamp  track_id   class  confidence     pixel_x      pixel_y
0      0   0.000000         0  person    0.813669  840.066498  1064.347046
1      0   0.000000         1  person    0.538242  872.067810   306.717812
2      1   0.016667         0  person    0.849687  840.448639  1063.881592
3      1   0.016667         1  person    0.573298  872.177307   306.921646
4      2   0.033333         0  person    0.833469  840.696259  1063.901947
5      2   0.033333         1  person    0.584119  872.178406   306.893204
6      3   0.050000         0  person    0.815066  841.014404  1063.839935
7      3   0.050000         1  person    0.603084  872.464050   306.983513
8      4   0.066667         0  person    0.820650  841.550201  1064.078949
9      4   0.066667         1  person    0.605688  872.76330

In [11]:
import pandas as pd
import json

# Load the tracking data
df = pd.read_csv("../outputs/demo_tracking_data.csv")

# Get video info
max_x = df['pixel_x'].max()
max_y = df['pixel_y'].max()
duration = df['timestamp'].max()

print(f"Video dimensions (approx): {max_x:.0f} x {max_y:.0f}")
print(f"Duration: {duration:.2f} seconds")
print(f"Total tracks: {df['track_id'].nunique()}")
print(f"Total detections: {len(df)}")

# Prepare data for JavaScript (group by frame)
frames_data = []
for frame_num in df['frame'].unique():
    frame_df = df[df['frame'] == frame_num]
    objects = []
    for _, row in frame_df.iterrows():
        objects.append({
            'track_id': int(row['track_id']),
            'class': row['class'],
            'x': float(row['pixel_x']),
            'y': float(row['pixel_y']),
            'confidence': float(row['confidence'])
        })
    frames_data.append({
        'frame': int(frame_num),
        'timestamp': float(frame_df.iloc[0]['timestamp']),
        'objects': objects
    })

# Save as JSON for the demo
with open('../outputs/tracking_data.json', 'w') as f:
    json.dump(frames_data, f, indent=2)

print(f"\nTracking data exported to: ../outputs/tracking_data.json")
print(f"Ready to create the demo!")

Video dimensions (approx): 1920 x 1080
Duration: 59.98 seconds
Total tracks: 50
Total detections: 11028

Tracking data exported to: ../outputs/tracking_data.json
Ready to create the demo!


In [22]:
import json
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d

input_file = "../outputs/tracking_data_20251204_231515.json"
output_file = "../outputs/tracking_data_cleaned_final.json"

print("Loading tracking data...")
with open(input_file, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} frames\n")

# Step 1: Define track mappings
keep_tracks = {
    0: 0,  # Keep as is
    66: 3  # Keep as is
}

# Tracks to merge (pick first ID as representative)
merge_group_1 = [21, 22, 26, 27, 28, 49, 51, 53, 54, 59]  # -> new id 1
merge_group_2 = [33, 34, 35]  # -> new id 2

for track_id in merge_group_1:
    keep_tracks[track_id] = 1
for track_id in merge_group_2:
    keep_tracks[track_id] = 2

print("Track mapping:")
print("  ID 0 -> 0 (person - constant)")
print(f"  IDs {merge_group_1} -> 1 (merged object)")
print(f"  IDs {merge_group_2} -> 2 (merged object)")
print("  ID 66 -> 3 (correct constantly)\n")

# Step 2: Filter and remap, handling duplicates per frame
print("Step 1: Filtering and merging tracks...")
filtered_data = []

for frame in data:
    frame_objects = {}  # new_id -> object data
    
    for obj in frame['objects']:
        old_id = obj['track_id']
        if old_id in keep_tracks:
            new_id = keep_tracks[old_id]
            
            # If this new_id already exists in this frame, keep the one with higher confidence
            if new_id in frame_objects:
                if obj['confidence'] > frame_objects[new_id]['confidence']:
                    frame_objects[new_id] = obj.copy()
                    frame_objects[new_id]['track_id'] = new_id
            else:
                frame_objects[new_id] = obj.copy()
                frame_objects[new_id]['track_id'] = new_id
    
    filtered_frame = {
        'frame': frame['frame'],
        'timestamp': frame['timestamp'],
        'objects': list(frame_objects.values())
    }
    filtered_data.append(filtered_frame)

print(f"  Filtered to {sum(len(f['objects']) for f in filtered_data)} detections\n")

# Step 3: Build complete trajectories for each track
print("Step 2: Building trajectories...")
trajectories = {0: {}, 1: {}, 2: {}, 3: {}}  # frame_num -> object data

for frame in filtered_data:
    frame_num = frame['frame']
    for obj in frame['objects']:
        track_id = obj['track_id']
        trajectories[track_id][frame_num] = obj

for tid in trajectories:
    print(f"  Track {tid}: {len(trajectories[tid])} frames")

# Step 4: Interpolate missing frames
print("\nStep 3: Interpolating missing frames...")

def interpolate_trajectory(traj_dict):
    if len(traj_dict) < 2:
        return traj_dict
    
    frame_nums = sorted(traj_dict.keys())
    min_frame = frame_nums[0]
    max_frame = frame_nums[-1]
    
    # Collect existing data
    frames = []
    pixel_x = []
    pixel_y = []
    world_x = []
    world_y = []
    
    for fn in frame_nums:
        frames.append(fn)
        pixel_x.append(traj_dict[fn]['pixel_x'])
        pixel_y.append(traj_dict[fn]['pixel_y'])
        world_x.append(traj_dict[fn]['world_x'])
        world_y.append(traj_dict[fn]['world_y'])
    
    # Create interpolation functions
    if len(frames) >= 2:
        interp_px = interp1d(frames, pixel_x, kind='linear', fill_value='extrapolate')
        interp_py = interp1d(frames, pixel_y, kind='linear', fill_value='extrapolate')
        interp_wx = interp1d(frames, world_x, kind='linear', fill_value='extrapolate')
        interp_wy = interp1d(frames, world_y, kind='linear', fill_value='extrapolate')
        
        # Fill in missing frames
        for fn in range(min_frame, max_frame + 1):
            if fn not in traj_dict:
                # Get template from nearest existing frame
                template = traj_dict[frame_nums[0]].copy()
                template['pixel_x'] = float(interp_px(fn))
                template['pixel_y'] = float(interp_py(fn))
                template['world_x'] = float(interp_wx(fn))
                template['world_y'] = float(interp_wy(fn))
                traj_dict[fn] = template
    
    return traj_dict

for track_id in trajectories:
    before = len(trajectories[track_id])
    trajectories[track_id] = interpolate_trajectory(trajectories[track_id])
    after = len(trajectories[track_id])
    if after > before:
        print(f"  Track {track_id}: Added {after - before} interpolated frames")

# Step 5: Smooth trajectories and remove outliers
print("\nStep 4: Smoothing and removing outliers...")

def smooth_and_fix_jumps(traj_dict, sigma=2):
    if len(traj_dict) < 5:
        return traj_dict
    
    frame_nums = sorted(traj_dict.keys())
    
    # Extract coordinates
    pixel_x = np.array([traj_dict[fn]['pixel_x'] for fn in frame_nums])
    pixel_y = np.array([traj_dict[fn]['pixel_y'] for fn in frame_nums])
    world_x = np.array([traj_dict[fn]['world_x'] for fn in frame_nums])
    world_y = np.array([traj_dict[fn]['world_y'] for fn in frame_nums])
    
    # Detect and fix outliers (sudden jumps)
    def fix_outliers(arr):
        # Calculate frame-to-frame differences
        diffs = np.abs(np.diff(arr))
        median_diff = np.median(diffs)
        threshold = median_diff * 5  # Jumps 5x larger than typical
        
        for i in range(1, len(arr)-1):
            if diffs[i-1] > threshold or diffs[i] > threshold:
                # Outlier detected, replace with average of neighbors
                arr[i] = (arr[i-1] + arr[i+1]) / 2
        return arr
    
    pixel_x = fix_outliers(pixel_x)
    pixel_y = fix_outliers(pixel_y)
    world_x = fix_outliers(world_x)
    world_y = fix_outliers(world_y)
    
    # Apply Gaussian smoothing
    pixel_x = gaussian_filter1d(pixel_x, sigma=sigma)
    pixel_y = gaussian_filter1d(pixel_y, sigma=sigma)
    world_x = gaussian_filter1d(world_x, sigma=sigma)
    world_y = gaussian_filter1d(world_y, sigma=sigma)
    
    # Update trajectory
    for i, fn in enumerate(frame_nums):
        traj_dict[fn]['pixel_x'] = float(pixel_x[i])
        traj_dict[fn]['pixel_y'] = float(pixel_y[i])
        traj_dict[fn]['world_x'] = float(world_x[i])
        traj_dict[fn]['world_y'] = float(world_y[i])
    
    return traj_dict

for track_id in trajectories:
    if len(trajectories[track_id]) > 0:
        trajectories[track_id] = smooth_and_fix_jumps(trajectories[track_id])
        print(f"  Track {track_id}: Smoothed and fixed jumps")

# Step 6: Reconstruct final data
print("\nStep 5: Reconstructing final data...")
final_data = []

for frame in filtered_data:
    frame_num = frame['frame']
    
    frame_objects = []
    for track_id in [0, 1, 2, 3]:
        if frame_num in trajectories[track_id]:
            frame_objects.append(trajectories[track_id][frame_num])
    
    final_frame = {
        'frame': frame_num,
        'timestamp': frame['timestamp'],
        'objects': frame_objects
    }
    final_data.append(final_frame)

# Step 7: Save
print("\nStep 6: Saving cleaned data...")
with open(output_file, 'w') as f:
    json.dump(final_data, f, indent=2)

print(f"\n✅ Cleaned data saved to: {output_file}")

# Statistics
total_detections = sum(len(f['objects']) for f in final_data)
print(f"\nFinal Statistics:")
print(f"  Total frames: {len(final_data)}")
print(f"  Total detections: {total_detections}")

for track_id in [0, 1, 2, 3]:
    count = sum(1 for f in final_data for obj in f['objects'] if obj['track_id'] == track_id)
    if count > 0:
        class_name = next((obj['class'] for f in final_data for obj in f['objects'] if obj['track_id'] == track_id), 'N/A')
        print(f"  Track {track_id} ({class_name}): {count} frames")

Loading tracking data...
Loaded 1000 frames

Track mapping:
  ID 0 -> 0 (person - constant)
  IDs [21, 22, 26, 27, 28, 49, 51, 53, 54, 59] -> 1 (merged object)
  IDs [33, 34, 35] -> 2 (merged object)
  ID 66 -> 3 (correct constantly)

Step 1: Filtering and merging tracks...
  Filtered to 2105 detections

Step 2: Building trajectories...
  Track 0: 1000 frames
  Track 1: 296 frames
  Track 2: 458 frames
  Track 3: 351 frames

Step 3: Interpolating missing frames...
  Track 1: Added 138 interpolated frames
  Track 2: Added 12 interpolated frames
  Track 3: Added 80 interpolated frames

Step 4: Smoothing and removing outliers...
  Track 0: Smoothed and fixed jumps
  Track 1: Smoothed and fixed jumps
  Track 2: Smoothed and fixed jumps
  Track 3: Smoothed and fixed jumps

Step 5: Reconstructing final data...

Step 6: Saving cleaned data...

✅ Cleaned data saved to: ../outputs/tracking_data_cleaned_final.json

Final Statistics:
  Total frames: 1000
  Total detections: 2335
  Track 0 (person

In [19]:
import sys
sys.path.append('src')

import cv2
import json
from object_detection.inference.predictor import ObjectDetector
from object_detection.tracking.simple_tracker import SimpleTracker

# Paths
video_path = "../outputs/trimmed_demo.mp4"
model_path = "../models/weights/best.pt"
output_json = "../outputs/tracking_export.json"

print("Loading model...")
detector = ObjectDetector(config_path=None, model_path=model_path)
detector.config.device = 'cpu'

print("Initializing tracker...")
tracker = SimpleTracker(max_disappeared=30)

# Open video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

# Storage for all tracking data
all_tracking_data = []

print(f"Processing video at {fps} fps...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run detection
    results = detector.predict(frame)
    detections = results[0]['detections'] if results and 'detections' in results[0] else []
    
    # Track objects
    tracked_objects = tracker.update(detections)
    
    # Store frame data
    frame_data = {
        'frame': frame_count,
        'timestamp': frame_count / fps,
        'objects': []
    }
    
    for obj in tracked_objects:
        bbox = obj['bbox']
        center_x = bbox[0] + bbox[2] / 2
        center_y = bbox[1] + bbox[3] / 2
        
        frame_data['objects'].append({
            'track_id': obj['track_id'],
            'class': obj['class_name'],
            'x': float(center_x),
            'y': float(center_y),
            'confidence': float(obj['confidence']),
            'bbox': [float(b) for b in bbox]
        })
    
    all_tracking_data.append(frame_data)
    
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()

# Save JSON
with open(output_json, 'w') as f:
    json.dump(all_tracking_data, f, indent=2)

print(f"\n✅ Tracking data saved to: {output_json}")
print(f"Total frames: {frame_count}")
print(f"Total detections: {sum(len(f['objects']) for f in all_tracking_data)}")

# Show track statistics
track_ids = set()
for frame in all_tracking_data:
    for obj in frame['objects']:
        track_ids.add(obj['track_id'])

print(f"Unique tracks: {len(track_ids)}")
print(f"Track IDs: {sorted(track_ids)}")


Loading model...
Initializing tracker...
Processing video at 60.0 fps...


0: 384x640 2 persons, 68.1ms
Speed: 2.3ms preprocess, 68.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3606

0: 384x640 2 persons, 62.7ms
Speed: 1.7ms preprocess, 62.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3607

0: 384x640 2 persons, 58.2ms
Speed: 1.3ms preprocess, 58.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3608

0: 384x640 2 persons, 60.7ms
Speed: 1.4ms preprocess, 60.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3609

0: 384x640 2 persons, 57.7ms
Speed: 1.5ms preprocess, 57.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3610

0: 384x640 2 persons, 59.8ms
Speed: 1.4ms preprocess, 59.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384,

Processed 30 frames...


0: 384x640 2 persons, 55.0ms
Speed: 1.4ms preprocess, 55.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3638

0: 384x640 2 persons, 58.6ms
Speed: 1.7ms preprocess, 58.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3639

0: 384x640 2 persons, 57.2ms
Speed: 1.2ms preprocess, 57.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3640

0: 384x640 2 persons, 55.4ms
Speed: 1.7ms preprocess, 55.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3641

0: 384x640 2 persons, 56.1ms
Speed: 1.3ms preprocess, 56.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict3642

0: 384x640 2 persons, 57.6ms
Speed: 1.3ms preprocess, 57.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384,

Processed 60 frames...


...

Processed 3540 frames...


0: 384x640 2 persons, 84.4ms
Speed: 1.2ms preprocess, 84.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7148

0: 384x640 2 persons, 52.4ms
Speed: 1.2ms preprocess, 52.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7149

0: 384x640 2 persons, 55.9ms
Speed: 1.3ms preprocess, 55.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7150

0: 384x640 2 persons, 52.0ms
Speed: 1.7ms preprocess, 52.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7151

0: 384x640 2 persons, 54.8ms
Speed: 1.4ms preprocess, 54.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7152

0: 384x640 2 persons, 53.1ms
Speed: 1.2ms preprocess, 53.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384,

Processed 3570 frames...


0: 384x640 2 persons, 55.4ms
Speed: 1.1ms preprocess, 55.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7178

0: 384x640 2 persons, 51.6ms
Speed: 1.1ms preprocess, 51.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7179

0: 384x640 2 persons, 53.6ms
Speed: 1.3ms preprocess, 53.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7180

0: 384x640 2 persons, 50.2ms
Speed: 1.4ms preprocess, 50.2ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7181

0: 384x640 2 persons, 53.7ms
Speed: 1.2ms preprocess, 53.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to outputs/predictions/predict7182

0: 384x640 2 persons, 50.7ms
Speed: 1.3ms preprocess, 50.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384,

Processed 3600 frames...

✅ Tracking data saved to: ../outputs/tracking_export.json
Total frames: 3600
Total detections: 11028
Unique tracks: 50
Track IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
